<div style="border-left:5px solid #67CBEE;padding:10px 16px;background:#F7F9FB;color:#19222D;margin:12px 0"><b>AMD Radeon · YOLO26 · Day 01</b><br><span style="color:#536273">算法 → 数据 → 训练 → 评测 → 证据。每一个结论都应由实际运行结果支撑。</span><br><span style="color:#536273">Algorithm → Data → Training → Evaluation → Evidence. Every conclusion must be backed by real execution results.</span></div>

# 00 / ROCm 环境预检与课程工作目录

**建议时长：10–15 分钟**　｜　学习路径：环境 → GPU 后端 → 持久化工件

> 请按顺序运行；运行结果必须来自自己的 GPU 环境，不可把示例文字当作实测数据。

---

# 00 / ROCm Environment Preflight & Course Workspace

**Suggested time: 10–15 minutes** | Learning path: environment → GPU backend → persistent artifacts

> Run the cells in order; every result must come from your own GPU environment — never present the sample text as measured data.

### 学习地图（30 秒看完这张表再开始）

| 问题 | 答案 |
|---|---|
| 为什么学这个？ | 环境是全部实验的地基——后面 5 个实验都靠它，错了全白跑 |
| 需要先懂什么？ | 会点 Jupyter 运行按钮即可，概念现场解释 |
| 我修改哪个参数？ | 练习格的 `SMOKE_N` 与 `MY_PREDICTION`（先预测再运行） |
| 成功长什么样？ | `torch_hip` 非空、GPU 名字打印、`HIP kernel smoke: PASS` |
| 失败怎么自救？ | 本页"常见错误与下一步"表 → `docs/TROUBLESHOOTING.md` |
| 保存什么证据？ | `reports/environment.json` + 权重 SHA256 |
| 求职价值？ | "给我一台 AMD GPU 服务器，我 5 分钟验明环境"——运维/算法岗都考 |

---

### Lab Map (read this 30-second table first)

| Question | Answer |
|---|---|
| Why learn this? | The environment is the foundation of all five later labs; a broken one wastes everything |
| Prerequisites? | Being able to run a Jupyter cell; concepts are explained on the spot |
| What do I change? | `SMOKE_N` and `MY_PREDICTION` in the practice cell (predict first, then run) |
| What does success look like? | Non-empty `torch_hip`, your GPU name printed, `HIP kernel smoke: PASS` |
| If it fails? | The common-errors table on this page → `docs/TROUBLESHOOTING.md` |
| Evidence to save? | `reports/environment.json` + the weight SHA256 |
| Job value? | "Hand me an AMD GPU server and I will verify the stack in 5 minutes" |

## 学习目标
- 确认当前 Jupyter Kernel 的 **PyTorch 是 ROCm/HIP 版本**，不是 CPU 或 NVIDIA CUDA 版。
- 检查 GPU 设备、Ultralytics 版本和 `/workspace` 持久化路径。
- 保存不含代理凭据的环境快照供实验复现。

**重要：** 不要在已安装 AMD 定制 PyTorch 的环境直接执行 `pip install torch` 或 `pip install -U torch`。如未安装 Ultralytics，请按照 [AMD 官方集成文档](https://docs.ultralytics.com/integrations/amd/) 在匹配的 ROCm 环境中安装。

---

## Learning Objectives
- Verify that the current Jupyter kernel runs a **ROCm/HIP build of PyTorch** — not CPU, not NVIDIA CUDA.
- Inspect the GPU device, the Ultralytics version and the `/workspace` persistence path.
- Save a proxy-credential-free environment snapshot for reproducibility.

**Important:** never run `pip install torch` or `pip install -U torch` inside an environment that already carries the AMD-customized PyTorch. If Ultralytics is missing, follow the [official AMD integration guide](https://docs.ultralytics.com/integrations/amd/) and install it into the matching ROCm environment.

In [1]:
from pathlib import Path
import os
import sys
import json

candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace/YOLO26_ROCm_Day1_Labs')]
package = next((p for p in candidates if (p / 'course/lab_utils.py').exists()), None)
if package is None:
    raise FileNotFoundError('请先将 ZIP 完整解压，并从项目目录或 notebooks/ 目录启动 Jupyter。')
sys.path.insert(0, str(package))
from course.lab_utils import (root, ensure_dirs, load_config, save_config, write_json,
                              write_csv, system_info, require_rocm, sha256)

home = ensure_dirs()
cfg = load_config()
print('Project:', package, '\nPersistent output:', home)

Project: /workspace/yolo26-rocm-day1-labs 
Persistent output: /workspace/yolo_course_release_final


In [2]:
from pprint import pprint

info = system_info()
info['gpu_device_nodes'] = {p: Path(p).exists() for p in ('/dev/kfd', '/dev/dri')}
info['proxy_configured'] = {n: bool(os.getenv(n)) for n in ('HTTP_PROXY', 'HTTPS_PROXY',
                                                            'http_proxy', 'https_proxy')}
write_json(home / 'reports/environment.json', info)
pprint(info)

{'gpu_device_nodes': {'/dev/dri': True, '/dev/kfd': True},
 'gpu_name': 'AMD Radeon Pro W7900D',
 'gpu_visible': True,
 'platform': 'Linux-6.8.0-79-generic-x86_64-with-glibc2.39',
 'proxy_configured': {'HTTPS_PROXY': True,
                      'HTTP_PROXY': True,
                      'http_proxy': True,
                      'https_proxy': True},
 'python': '3.12.13',
 'timestamp_utc': '2026-11-03T03:50:52Z',
 'torch': '2.12.0+rocm7.14.1',
 'torch_hip': '7.14.60850',
 'ultralytics': '8.4.199'}


### 理解环境信息
- `torch.version.hip != None` 表示 ROCm PyTorch 构建。
- 在 AMD GPU 上，`torch.cuda.is_available()` 也可以返回 `True`，这是 HIP 对 PyTorch CUDA API 的兼容设计。
- 训练传 `device=0`；**不要**传 `device="rocm"`。
- 云端代理只需要继承平台设置；本课不会输出代理 URL、访问令牌或秘密。

**依赖**：这是第一份 Notebook，无前置实验；只需按部署文档建好 ROCm 环境。

### 预期输出（正常长什么样）
- `environment.json` 打印出的字典里：`torch_hip` 是一串版本号（非空）、`gpu_name` 是你的 AMD 显卡名、`gpu_device_nodes` 两项都是 `True`。
- ROCm 检查格打印 `AMD ROCm verified: <你的显卡名> HIP: <版本>`。
- HIP 冒烟格打印 `HIP kernel smoke: PASS, output shape: (128, 128)`。
- 若与上述形状不符（例如 `torch_hip` 为 `None` 或 GPU 不可见），先看下面的常见错误，不要继续后面的实验。

### 常见错误与下一步
| 症状 | 原因 | 下一步 |
|---|---|---|
| `torch.version.hip` 为 `None` | 当前 Kernel 不是 ROCm 构建（CPU/CUDA wheel） | 切换到课程注册的 Kernel；重装见部署文档 |
| `no GPU visible` / `/dev/kfd` 不存在 | GPU 未分配或节点不可见 | 检查平台 GPU 分配；Linux 看 `/dev/kfd`、`/dev/dri` |
| `torchvision GPU NMS is unavailable` | 平台预装 torchvision 缺 GPU NMS 内核 | 按部署文档方式 B 自建 uv 隔离栈 |
| `pip install torch` 后环境损坏 | 覆盖了 AMD 定制 PyTorch | 重建环境，永远不要在课程环境里装默认 torch |

更多排障见 `docs/TROUBLESHOOTING.md`。

---

### Reading the Environment Info
- `torch.version.hip != None` marks a ROCm build of PyTorch.
- On AMD GPUs `torch.cuda.is_available()` may also return `True` — that is HIP's compatibility layer over the PyTorch CUDA API surface.
- Pass `device=0` for training; **do not** pass `device="rocm"`.
- Cloud proxies only need the platform defaults; this course never prints proxy URLs, tokens or secrets.

**Prerequisite**: this is the first notebook — no prior lab; only a working ROCm environment from the setup guide.

### Expected Output (what healthy looks like)
- In the printed `environment.json` dict: `torch_hip` is a non-empty version string, `gpu_name` is your AMD GPU, and both `gpu_device_nodes` entries are `True`.
- The ROCm gate cell prints `AMD ROCm verified: <your GPU> HIP: <version>`.
- The HIP smoke cell prints `HIP kernel smoke: PASS, output shape: (128, 128)`.
- If the shape differs (e.g. `torch_hip` is `None` or no GPU), read the common errors below before continuing.

### Common Errors & Next Steps
| Symptom | Cause | Next step |
|---|---|---|
| `torch.version.hip` is `None` | kernel is not a ROCm build (CPU/CUDA wheel) | switch to the registered course kernel; reinstall per the setup guide |
| `no GPU visible` / missing `/dev/kfd` | GPU not allocated or nodes hidden | check platform GPU assignment; on Linux inspect `/dev/kfd`, `/dev/dri` |
| `torchvision GPU NMS is unavailable` | platform torchvision lacks the GPU NMS kernel | build the uv-isolated stack (setup guide, option B) |
| environment broken after `pip install torch` | AMD-customized PyTorch overwritten | rebuild the env; never install default torch inside it |

Full troubleshooting: `docs/TROUBLESHOOTING.md`.

In [3]:
STRICT_ROCM = True  # 教学 GPU 实验应保持 True；不要为了通过检查而改成 False
if STRICT_ROCM:
    device = require_rocm()
else:
    device = 'cpu'
    print('CPU-only diagnostic mode; GPU experiment conclusions are NOT valid.')
print('Device argument for Ultralytics:', device)

AMD ROCm verified: AMD Radeon Pro W7900D HIP: 7.14.60850
Device argument for Ultralytics: 0


In [4]:
# 可选：实际在 GPU 上计算一个小矩阵，验证张量确实被送入 GPU
import torch

if STRICT_ROCM:
    x = torch.randn((128, 128), device='cuda:0')
    y = x @ x
    torch.cuda.synchronize()
    assert y.device.type == 'cuda'  # ROCm 也报告 cuda device type
    print('HIP kernel smoke: PASS, output shape:', tuple(y.shape))

HIP kernel smoke: PASS, output shape: (128, 128)


### 练习：先预测，再运行（5 分钟）
计算思维的第一步是**预测→运行→对照**。下面这个格子默认能跑通，但直接跑没有意义：
1. 把 `SMOKE_N` 改成 256（或任意正整数）。
2. **先**把你的预测写进 `MY_PREDICTION`（替换两个 TODO）。
3. 运行，对照 `actual` 输出；猜错完全没关系——关键是养成先预测的习惯。
你的预测会保存到 `reports/lab00_practice.json`，作为你动手的证据。

---

### Try It Yourself: predict, then run (5 min)
Computational thinking starts with **predict → run → compare**. The cell below runs as-is, but running it blindly teaches nothing:
1. Change `SMOKE_N` to 256 (or any positive integer).
2. **First** write your prediction into `MY_PREDICTION` (replace both TODOs).
3. Run it and compare with the `actual` output; being wrong is fine — the habit of predicting first is the point.
Your prediction is saved to `reports/lab00_practice.json` as evidence of hands-on work.

In [5]:
# 学生练习：预测 → 运行 → 对照（修改 SMOKE_N 与 MY_PREDICTION 后运行）
SMOKE_N = 128
MY_PREDICTION = {'shape': 'TODO: 写下你预测的输出形状，如 (128, 128)',
                 'device': 'TODO: 你预测结果张量落在哪个设备？'}

# 无人值守复验通道（学生不需要）：设 YOLO_PRACTICE_FILL 指向 JSON 文件可注入已填写样例，
# 注入值会原样出现在本格输出与保存的报告里——不做任何隐藏。
if os.environ.get('YOLO_PRACTICE_FILL'):
    _p = Path(os.environ['YOLO_PRACTICE_FILL'])
    if _p.is_file():
        for _k, _v in json.loads(_p.read_text(encoding='utf-8')).get('lab00_practice', {}).items():
            MY_PREDICTION[_k] = _v

import torch

for key, value in MY_PREDICTION.items():
    assert not str(value).startswith('TODO'), f'请先填写 {key} 的预测再运行——直接运行无法形成预测习惯'
probe = torch.randn((SMOKE_N, SMOKE_N), device='cuda:0')
outcome = probe @ probe
torch.cuda.synchronize()
print('actual shape:', tuple(outcome.shape), '| actual device:', outcome.device)
matched = str(MY_PREDICTION['shape']).replace(' ', '') == str(tuple(outcome.shape))
print('your prediction:', MY_PREDICTION)
print('prediction matched:', matched, '（猜错不扣分；先预测再验证才是目的）')
write_json(home / 'reports/lab00_practice.json',
           {'smoke_n': SMOKE_N, 'prediction': MY_PREDICTION,
            'actual': {'shape': list(outcome.shape), 'device': str(outcome.device)},
            'matched': matched})

actual shape: (128, 128) | actual device: cuda:0
your prediction: {'shape': '(128, 128)', 'device': 'cuda:0 (HIP)'}
prediction matched: False （猜错不扣分；先预测再验证才是目的）


### 冻结预训练权重（所有 A/C/E 使用相同模型）
为了避免课程中途的自动下载或不同版本导致 A/B 实验起点不一致，首次联网时缓存 `yolo26n.pt`，后续统一从 `/workspace/yolo_course/artifacts/yolo26n.pt` 读取。缓存成功后记录 SHA256。

---

### Pinning the Pretrained Weights (labs A/C/E share one model)
To keep every A/B experiment starting from an identical point — and to avoid mid-class downloads — cache `yolo26n.pt` on first network use, then always load it from `/workspace/yolo_course/artifacts/yolo26n.pt`. Its SHA256 is recorded once the cache is pinned.

In [6]:
import shutil
from ultralytics import YOLO

model_cache = home / 'artifacts/yolo26n.pt'
if not model_cache.is_file():
    fresh = YOLO('yolo26n.pt')
    source = Path(fresh.ckpt_path).resolve()
    assert source.is_file(), f'Model download not found at {source}'
    if source != model_cache.resolve():
        shutil.copy2(source, model_cache)
assert model_cache.is_file()
print('Pinned model:', model_cache, 'sha256:', sha256(model_cache))
save_config({'model': 'yolo26n.pt', 'model_sha256': sha256(model_cache)})

Pinned model: /workspace/yolo_course_release_final/artifacts/yolo26n.pt sha256: 9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef


{'home': '/workspace/yolo_course_release_final',
 'model': 'yolo26n.pt',
 'dataset_yaml': '',
 'dataset_kind': '',
 'baseline_weights': '',
 'baseline_run': '',
 'model_licence_reviewed': False,
 'model_sha256': '9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef'}

### 课后问题（写进环境报告）
1. 为什么 AMD GPU 使用 `torch.cuda`？
2. `torch.version.hip` 返回 `None` 意味着什么？
3. 为什么不能在已经配好的 ROCm 环境随意覆盖 PyTorch？

**面试可能这样问**：入职后给你一台 AMD GPU 服务器，你怎么确认上面的深度学习环境能用？会检查哪些字段、分别排除什么问题？

**带走什么 / Takeaway**：环境是实验的地基——`torch.version.hip`、GPU 可见性、设备节点、权重指纹（SHA256），四样确认完才开始训练。

**验收文件**：`reports/environment.json`、`reports/lab00_practice.json`。

---

### After-Class Questions (put answers in your environment report)
1. Why do AMD GPUs use the `torch.cuda` API?
2. What does it mean when `torch.version.hip` returns `None`?
3. Why must you never overwrite PyTorch in an already-configured ROCm environment?

**Interview-style question**: you are handed an AMD GPU server on day one — how do you verify its deep-learning stack works? Which fields do you check, and what does each rule out?

**Takeaway**: the environment is the foundation of every experiment — `torch.version.hip`, GPU visibility, device nodes, and the weight fingerprint (SHA256). Confirm all four before any training.

**Acceptance artifacts**: `reports/environment.json`, `reports/lab00_practice.json`.